# TwHIN на Yelp

Воспроизведение TwHIN (El-Kishky et al., KDD 2022) на открытом Yelp Open Dataset.

- Скоринг: dot-product `f(s,r,t) = (θ_s + θ_r)·θ_t`
- Лосс: negative sampling или sampled-softmax (+ logQ-коррекция)
- Негативы: порча source и target, того же типа сущности; uniform или по частоте
- Оптимизатор: Adagrad
- Mixture-of-embeddings
- Метрики: Recall@K/MRR, RCE, PR-AUC

## 0. Конфиг

In [ ]:
from pathlib import Path

CFG = {
    # данные
    "data_dir": "/home/nikita/DL/recsys/yelp",
    "out_dir": "outputs/yelp_graph",
    "ckpt_dir": "outputs/checkpoints",
    "state": "PA",  # фильтр по штату
    "train_end": "2020-12-31",
    "val_end": "2021-06-30",
    "friend_test_frac": 0.1,
    "seed": 42,
    # модель / обучение
    "embedding_dim": 128,
    "batch_size": 4096,
    "epochs": 20,
    "lr": 0.05,
    "weight_decay": 1.0e-05,
    "negative_samples": 64,
    "grad_clip_norm": 1.0,
    "mixed_precision": True,
    "num_workers": 0,
    "neg_strategy": "frequency",
    "neg_power": 0.75,
    "corrupt": "both",
    # оценка
    "recall_ks": [10, 20, 50],
    "mixture_clusters": 1000,
    "mixture_top_m": 3,
    "early_stop_patience": 5,
}

for k in ("out_dir", "ckpt_dir"):
    Path(CFG[k]).mkdir(parents=True, exist_ok=True)
CFG

{'data_dir': '/home/nikita/DL/recsys/yelp',
 'out_dir': 'outputs/yelp_graph',
 'ckpt_dir': 'outputs/checkpoints',
 'state': 'PA',
 'train_end': '2020-12-31',
 'val_end': '2021-06-30',
 'friend_test_frac': 0.1,
 'seed': 42,
 'embedding_dim': 128,
 'batch_size': 4096,
 'epochs': 20,
 'lr': 0.05,
 'weight_decay': 1e-05,
 'negative_samples': 64,
 'grad_clip_norm': 1.0,
 'mixed_precision': True,
 'num_workers': 0,
 'neg_strategy': 'frequency',
 'neg_power': 0.75,
 'corrupt': 'both',
 'recall_ks': [10, 20, 50],
 'mixture_clusters': 1000,
 'mixture_top_m': 3,
 'early_stop_patience': 5}

## 1. Препроцессинг

In [2]:
import json
import numpy as np
import pandas as pd

DATA = Path(CFG["data_dir"])
OUT = Path(CFG["out_dir"])

def read_jsonl(path):
    with open(path) as f:
        for line in f:
            yield json.loads(line)

biz = []
for b in read_jsonl(DATA / "yelp_academic_dataset_business.json"):
    if b.get("state") == CFG["state"]:
        biz.append(b["business_id"])
biz_ids = set(biz)
print(f"businesses in {CFG['state']}: {len(biz_ids):,}")

businesses in PA: 34,039


In [3]:
reviews = [
    {"user_id": r["user_id"], "business_id": r["business_id"], "date": r["date"], "useful": r.get("useful", 0)}
    for r in read_jsonl(DATA / "yelp_academic_dataset_review.json") if r["business_id"] in biz_ids
]
review_df = pd.DataFrame(reviews)

tips = [
    {"user_id": t["user_id"], "business_id": t["business_id"], "date": t["date"]}
    for t in read_jsonl(DATA / "yelp_academic_dataset_tip.json") if t["business_id"] in biz_ids
]
tip_df = pd.DataFrame(tips)
print(f"reviews: {len(review_df):,}   tips: {len(tip_df):,}")

reviews: 1,598,960   tips: 193,609


In [4]:
active_users = set(review_df["user_id"]) | set(tip_df["user_id"])

friend_pairs = []
for u in read_jsonl(DATA / "yelp_academic_dataset_user.json"):
    if u["user_id"] not in active_users:
        continue
    fr = u.get("friends", "")
    if fr and fr != "None":
        for v in fr.split(", "):
            v = v.strip()
            if v in active_users and u["user_id"] < v:
                friend_pairs.append((u["user_id"], v))
friends_df = pd.DataFrame(friend_pairs, columns=["u", "v"]).drop_duplicates()
print(f"users: {len(active_users):,}   friend pairs: {len(friends_df):,}")

users: 434,918   friend pairs: 1,271,665


In [5]:
users_sorted = sorted(active_users)
biz_sorted = sorted(biz_ids & (set(review_df["business_id"]) | set(tip_df["business_id"])))

user2id = {u: i for i, u in enumerate(users_sorted)}
n_users = len(user2id)
biz2id = {b: n_users + i for i, b in enumerate(biz_sorted)}
n_biz = len(biz2id)
n_ent = n_users + n_biz

REL = {"review": 0, "tip": 1, "friend": 2}
REL_INFO = {
    0: {"name": "review", "lhs": "user", "rhs": "business", "coverage": "high"},
    1: {"name": "tip", "lhs": "user", "rhs": "business", "coverage": "low"},
    2: {"name": "friend", "lhs": "user", "rhs": "user", "coverage": "high"},
}
print(f"entities: {n_ent:,}  (users={n_users:,}, businesses={n_biz:,})")

entities: 468,957  (users=434,918, businesses=34,039)


In [6]:
def temporal_split(df):
    d = pd.to_datetime(df["date"])
    tr = pd.Timestamp(CFG["train_end"]); va = pd.Timestamp(CFG["val_end"])
    return df[d <= tr], df[(d > tr) & (d <= va)], df[d > va]

def ub_edges(df, rel):
    return pd.DataFrame({
        "src": df["user_id"].map(user2id),
        "rel": rel,
        "dst": df["business_id"].map(biz2id),
    }).dropna().astype(int)

rev_tr, rev_va, rev_te = temporal_split(review_df)
tip_tr, tip_va, tip_te = temporal_split(tip_df)

review_train, review_val, review_test = ub_edges(rev_tr, REL["review"]), ub_edges(rev_va, REL["review"]), ub_edges(rev_te, REL["review"])
tip_train, tip_val, tip_test = ub_edges(tip_tr, REL["tip"]), ub_edges(tip_va, REL["tip"]), ub_edges(tip_te, REL["tip"])

In [7]:
fr = pd.DataFrame({
    "src": friends_df["u"].map(user2id),
    "rel": REL["friend"],
    "dst": friends_df["v"].map(user2id),
}).dropna().astype(int)

rng = np.random.default_rng(CFG["seed"])
perm = rng.permutation(len(fr))
n_ft = int(len(fr) * CFG["friend_test_frac"])
fr_test_oneway = fr.iloc[perm[:n_ft]].reset_index(drop=True)
fr_train_oneway = fr.iloc[perm[n_ft:]].reset_index(drop=True)

# friend симметричен: train дублируем в обе стороны; test оставляем как hold-out пары
fr_rev = fr_train_oneway.rename(columns={"src": "dst", "dst": "src"})[["src", "rel", "dst"]]
friend_train = pd.concat([fr_train_oneway, fr_rev], ignore_index=True)

train_users = set(friend_train["src"]) | set(friend_train["dst"])
friend_test = fr_test_oneway[
    fr_test_oneway["src"].isin(train_users) & fr_test_oneway["dst"].isin(train_users)
].reset_index(drop=True)
print(f"friend train (2-way): {len(friend_train):,}   friend test: {len(friend_test):,}")

friend train (2-way): 2,288,998   friend test: 122,985


In [8]:
def save_tsv(df, path):
    df[["src", "rel", "dst"]].to_csv(path, sep="\t", index=False, header=False)

train_all = pd.concat([review_train, tip_train, friend_train], ignore_index=True)
val_all = pd.concat([review_val, tip_val], ignore_index=True)
test_all = pd.concat([review_test, tip_test], ignore_index=True)

for df, nm in [(train_all, "train.tsv"), (val_all, "val.tsv"), (test_all, "test.tsv"),
               (review_train, "review_train.tsv"), (review_test, "review_test.tsv"),
               (tip_train, "tip_train.tsv"),
               (friend_train, "friend_train.tsv"), (friend_test, "friend_test.tsv")]:
    save_tsv(df, OUT / nm)

ablations = {
    "all": [review_train, tip_train, friend_train],
    "no_tip": [review_train, friend_train],
    "no_friend": [review_train, tip_train],
    "review_only": [review_train],
    "friend_only": [friend_train],
    "review_friend": [review_train, friend_train],
}
for name, parts in ablations.items():
    d = OUT / f"ablation_{name}"; d.mkdir(exist_ok=True)
    save_tsv(pd.concat(parts, ignore_index=True), d / "train.tsv")
    save_tsv(val_all, d / "val.tsv"); save_tsv(test_all, d / "test.tsv")

ent_rows = [{"entity_id": i, "original_id": u, "entity_type": "user"} for u, i in user2id.items()]
ent_rows += [{"entity_id": i, "original_id": b, "entity_type": "business"} for b, i in biz2id.items()]
pd.DataFrame(ent_rows).to_parquet(OUT / "entities.parquet", index=False)
pd.DataFrame([{"relation_id": k, **v} for k, v in REL_INFO.items()]).to_parquet(OUT / "relations.parquet", index=False)

report = {
    "entities": {"user": n_users, "business": n_biz, "total": n_ent},
    "edges": {"review_train": len(review_train), "tip_train": len(tip_train),
              "friend_train": len(friend_train), "friend_test": len(friend_test),
              "review_test": len(review_test), "total_train": len(train_all)},
    "ablations": list(ablations.keys()),
}
json.dump(report, open(OUT / "split_report.json", "w"), indent=2)
report

{'entities': {'user': 434918, 'business': 34039, 'total': 468957},
 'edges': {'review_train': 1490530,
  'tip_train': 187331,
  'friend_train': 2288998,
  'friend_test': 122985,
  'review_test': 59496,
  'total_train': 3966859},
 'ablations': ['all',
  'no_tip',
  'no_friend',
  'review_only',
  'friend_only',
  'review_friend']}

### Проверки графа

In [9]:
tr = pd.read_csv(OUT / "train.tsv", sep="\t", header=None, names=["s", "r", "d"])
assert tr["s"].max() < n_ent and tr["d"].max() < n_ent, "ID вне словаря"
assert tr[["s", "d"]].min().min() >= 0

ft = pd.read_csv(OUT / "friend_test.tsv", sep="\t", header=None, names=["s", "r", "d"])
tr_fr = friend_train
train_pairs = set(map(frozenset, zip(tr_fr["src"], tr_fr["dst"])))
test_pairs = set(map(frozenset, zip(ft["s"], ft["d"])))
assert not (train_pairs & test_pairs), "friend_test пересекается с train (утечка)"
print("проверки графа пройдены")
print("рёбер по типам в train:", tr["r"].map({v: k for k, v in REL.items()}).value_counts().to_dict())

проверки графа пройдены
рёбер по типам в train: {'friend': 2288998, 'review': 1490530, 'tip': 187331}


## 2. Модель

In [10]:
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(CFG["seed"])
print("device:", DEVICE)

class TransE(nn.Module):
    def __init__(self, n_entities, n_relations, dim):
        super().__init__()
        self.ent = nn.Embedding(n_entities, dim)
        self.rel = nn.Embedding(n_relations, dim)
        nn.init.xavier_uniform_(self.ent.weight)
        nn.init.xavier_uniform_(self.rel.weight)

    def score(self, h, r, t):
        return ((self.ent(h) + self.rel(r)) * self.ent(t)).sum(-1)

    def tail_scores(self, h, r, cand_t):
        q = (self.ent(h) + self.rel(r))[:, None, :]
        return (q * self.ent(cand_t)).sum(-1)

    def head_scores(self, cand_h, r, t):
        q = (self.ent(t) - self.rel(r))[:, None, :]
        return (q * self.ent(cand_h)).sum(-1)

device: cuda


## 3. Типизированный негатив-сэмплер

In [ ]:
class TypedSampler:
    def __init__(self, train_tsv, n_users, n_ent, strategy, power):
        self.strategy = strategy
        self.user_ids = torch.arange(0, n_users)
        self.biz_ids = torch.arange(n_users, n_ent)
        self.n_users, self.n_ent = n_users, n_ent
        if strategy == "frequency":
            df = pd.read_csv(train_tsv, sep="\t", header=None, names=["s", "r", "d"])
            cnt = np.ones(n_ent)
            vc = pd.concat([df["s"], df["d"]]).value_counts()
            cnt[vc.index.to_numpy()] = vc.to_numpy()
            cnt = cnt ** power
            self.pu = torch.tensor(cnt[:n_users] / cnt[:n_users].sum(), dtype=torch.float)
            self.pb = torch.tensor(cnt[n_users:] / cnt[n_users:].sum(), dtype=torch.float)
            full = cnt / cnt.sum()
            self.logq = torch.tensor(np.log(full + 1e-12), dtype=torch.float)
        else:
            self.logq = torch.full((n_ent,), -np.log(n_ent), dtype=torch.float)

    def _sample_type(self, is_user, shape, device, generator):
        n = int(np.prod(shape))
        if self.strategy == "uniform":
            ids = (self.user_ids if is_user else self.biz_ids)
            idx = torch.randint(0, len(ids), (n,), generator=generator)
            return ids[idx].to(device).view(*shape)
        p = self.pu if is_user else self.pb
        base = self.user_ids if is_user else self.biz_ids
        idx = torch.multinomial(p, n, replacement=True, generator=generator)
        return base[idx].to(device).view(*shape)

    def sample_like(self, ref_ids, k, device, generator=None):
        is_user = (ref_ids < self.n_users)
        out = torch.empty((ref_ids.size(0), k), dtype=torch.long, device=device)
        if is_user.any():
            out[is_user] = self._sample_type(True, (int(is_user.sum()), k), device, generator)
        if (~is_user).any():
            out[~is_user] = self._sample_type(False, (int((~is_user).sum()), k), device, generator)
        return out

    def log_q(self, ids, device):
        return self.logq.to(device)[ids]

## 4. Лоссы: negative sampling, sampled-softmax, sampled-softmax + logQ

In [12]:
import torch.nn.functional as F

def ns_loss(pos, neg):
    return -(F.logsigmoid(pos) + F.logsigmoid(-neg).mean(1)).mean()

def sampled_softmax_loss(pos, neg, log_q=None):
    logits = torch.cat([pos[:, None], neg], dim=1)
    if log_q is not None:
        logits = logits - log_q.clamp(min=-20.0)
    target = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
    return F.cross_entropy(logits, target)

def compute_loss(model, h, r, t, sampler, neg_k, loss_name, corrupt, use_logq, generator=None):
    scale = model.ent.weight.size(1) ** 0.5
    pos = model.score(h, r, t)
    negs, neg_ids = [], []
    if corrupt in ("tail", "both"):
        k = neg_k // 2 if corrupt == "both" else neg_k
        nt = sampler.sample_like(t, k, h.device, generator)
        negs.append(model.tail_scores(h, r, nt)); neg_ids.append(nt)
    if corrupt in ("head", "both"):
        k = neg_k - neg_k // 2 if corrupt == "both" else neg_k
        nh = sampler.sample_like(h, k, h.device, generator)
        negs.append(model.head_scores(nh, r, t)); neg_ids.append(nh)
    neg = torch.cat(negs, 1)
    if loss_name == "ns":
        return ns_loss(pos, neg)
    log_q = None
    if use_logq:
        cand = torch.cat(neg_ids, 1)
        pos_t = t if corrupt != "head" else h
        log_q = torch.cat([sampler.log_q(pos_t[:, None], h.device), sampler.log_q(cand, h.device)], 1)
    return sampled_softmax_loss(pos / scale, neg / scale, log_q)

## 5. Обучение

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm
import faiss

def make_loader(tsv, shuffle):
    df = pd.read_csv(tsv, sep="\t", header=None, names=["s", "r", "d"])
    ds = TensorDataset(torch.tensor(df["s"].values), torch.tensor(df["r"].values), torch.tensor(df["d"].values))
    return DataLoader(ds, batch_size=CFG["batch_size"], shuffle=shuffle,
                      num_workers=CFG["num_workers"], pin_memory=(DEVICE.type == "cuda"), drop_last=shuffle)

@torch.no_grad()
def eval_loss(model, loader, sampler, loss_name, corrupt, use_logq):
    model.eval()
    g = torch.Generator(device="cpu"); g.manual_seed(123)
    tot = []
    for h, r, t in loader:
        h, r, t = h.to(DEVICE), r.to(DEVICE), t.to(DEVICE)
        tot.append(compute_loss(model, h, r, t, sampler, CFG["negative_samples"], loss_name, corrupt, use_logq, g).item())
    return float(np.mean(tot))

def _load_friend_eval():
    gt = {}
    for s, _, d in pd.read_csv(OUT / "friend_test.tsv", sep="\t", header=None).itertuples(index=False):
        gt.setdefault(int(s), set()).add(int(d))
    known = {}
    for s, _, d in pd.read_csv(OUT / "friend_train.tsv", sep="\t", header=None).itertuples(index=False):
        known.setdefault(int(s), set()).add(int(d)); known.setdefault(int(d), set()).add(int(s))
    return gt, known

_FRIEND_GT, _FRIEND_KNOWN = None, None

@torch.no_grad()
def quick_recall(model, k=10, max_q=5000):
    # unimodal Recall@k на friend hold-out
    global _FRIEND_GT, _FRIEND_KNOWN
    if _FRIEND_GT is None:
        _FRIEND_GT, _FRIEND_KNOWN = _load_friend_eval()
    model.eval()
    emb = model.ent.weight.detach().cpu().numpy()[:n_users].astype("float32")
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True).clip(min=1e-12)
    index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
    qs = [u for u in _FRIEND_GT if u < n_users][:max_q]
    Q = np.stack([emb[u] for u in qs])
    _, I = index.search(Q, k + 60)
    hits = []
    for u, row in zip(qs, I):
        seen = _FRIEND_KNOWN.get(u, set())
        cand = [int(x) for x in row if x != u and x not in seen][:k]
        hits.append(1.0 if set(cand) & _FRIEND_GT[u] else 0.0)
    return float(np.mean(hits)) if hits else 0.0

def train(ablation="all", loss_name="ns", use_logq=False, neg_strategy=None):
    neg_strategy = neg_strategy or CFG["neg_strategy"]
    adir = OUT / f"ablation_{ablation}"
    train_loader = make_loader(adir / "train.tsv", True)
    sampler = TypedSampler(adir / "train.tsv", n_users, n_ent, neg_strategy, CFG["neg_power"])

    model = TransE(n_ent, len(REL_INFO), CFG["embedding_dim"]).to(DEVICE)
    opt = torch.optim.Adagrad(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    amp = CFG["mixed_precision"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    tag = f"{ablation}__{loss_name}" + ("__logq" if use_logq else "") + f"__{neg_strategy}"
    best_r, curve, bad = -1.0, [], 0
    for ep in range(1, CFG["epochs"] + 1):
        model.train(); run = 0.0
        pbar = tqdm(train_loader, desc=f"{tag} ep{ep}")
        for h, r, t in pbar:
            h, r, t = h.to(DEVICE), r.to(DEVICE), t.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=amp):
                loss = compute_loss(model, h, r, t, sampler, CFG["negative_samples"], loss_name, CFG["corrupt"], use_logq)
            scaler.scale(loss).backward()
            if CFG["grad_clip_norm"] > 0:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip_norm"])
            scaler.step(opt); scaler.update()
            run += loss.item(); pbar.set_postfix(loss=f"{run/(pbar.n+1):.4f}")
        rec = quick_recall(model, k=CFG["recall_ks"][0])
        curve.append(rec); print(f"  ep{ep} train_loss={run/len(train_loader):.4f}  val_Recall@{CFG['recall_ks'][0]}={rec:.4f}")
        if rec > best_r:
            best_r = rec; bad = 0
            torch.save({"state_dict": model.state_dict(), "n_ent": n_ent, "n_rel": len(REL_INFO),
                        "dim": CFG["embedding_dim"], "tag": tag, "val_recall": rec}, Path(CFG["ckpt_dir"]) / f"{tag}.pt")
        else:
            bad += 1
            if bad >= CFG["early_stop_patience"]:
                print(f"  early stop на эпохе {ep} (нет роста Recall {bad} эпох)"); break
    print(f"  best val_Recall@{CFG['recall_ks'][0]}={best_r:.4f}")
    return tag, curve

In [14]:
tag_ns, curve_ns = train(ablation="all", loss_name="ns")
print("OK", tag_ns, "best val_Recall@10 =", round(max(curve_ns), 4))

all__ns__frequency ep1:   0%|          | 0/968 [00:00<?, ?it/s]

  ep1 train_loss=1.1387  val_Recall@10=0.0314


all__ns__frequency ep2:   0%|          | 0/968 [00:00<?, ?it/s]

  ep2 train_loss=1.0892  val_Recall@10=0.0298


all__ns__frequency ep3:   0%|          | 0/968 [00:00<?, ?it/s]

...

  ep14 train_loss=1.0439  val_Recall@10=0.0336


all__ns__frequency ep15:   0%|          | 0/968 [00:00<?, ?it/s]

  ep15 train_loss=1.0430  val_Recall@10=0.0342


all__ns__frequency ep16:   0%|          | 0/968 [00:00<?, ?it/s]

  ep16 train_loss=1.0423  val_Recall@10=0.0300
  early stop на эпохе 16 (нет роста Recall 5 эпох)
  best val_Recall@10=0.0362
OK all__ns__frequency best val_Recall@10 = 0.0362


In [15]:
tag_ssl, curve_ssl = train(ablation="all", loss_name="sampled_softmax", use_logq=True)
print("OK", tag_ssl, round(curve_ssl[0], 3), "->", round(curve_ssl[-1], 3))

all__sampled_softmax__logq__frequency ep1:   0%|          | 0/968 [00:00<?, ?it/s]

  ep1 train_loss=3.9491  val_Recall@10=0.0156


all__sampled_softmax__logq__frequency ep2:   0%|          | 0/968 [00:00<?, ?it/s]

  ep2 train_loss=3.6520  val_Recall@10=0.0202


all__sampled_softmax__logq__frequency ep3:   0%|          | 0/968 [00:00<?, ?it/s]

...

  ep10 train_loss=3.4803  val_Recall@10=0.0262


all__sampled_softmax__logq__frequency ep11:   0%|          | 0/968 [00:00<?, ?it/s]

  ep11 train_loss=3.4740  val_Recall@10=0.0242


all__sampled_softmax__logq__frequency ep12:   0%|          | 0/968 [00:00<?, ?it/s]

  ep12 train_loss=3.4682  val_Recall@10=0.0256
  early stop на эпохе 12 (нет роста Recall 5 эпох)
  best val_Recall@10=0.0262
OK all__sampled_softmax__logq__frequency 0.016 -> 0.026


## 6. Оценка

In [16]:
import faiss
from sklearn.cluster import MiniBatchKMeans

def load_emb(tag):
    ck = torch.load(Path(CFG["ckpt_dir"]) / f"{tag}.pt", map_location="cpu")
    m = TransE(ck["n_ent"], ck["n_rel"], ck["dim"]); m.load_state_dict(ck["state_dict"])
    e = m.ent.weight.detach().numpy().astype("float32")
    return e / np.linalg.norm(e, axis=1, keepdims=True).clip(min=1e-12)

def load_gt_known():
    gt = {}
    for s, _, d in pd.read_csv(OUT / "friend_test.tsv", sep="\t", header=None).itertuples(index=False):
        gt.setdefault(int(s), set()).add(int(d))
    known = {}
    for s, _, d in pd.read_csv(OUT / "friend_train.tsv", sep="\t", header=None).itertuples(index=False):
        known.setdefault(int(s), set()).add(int(d)); known.setdefault(int(d), set()).add(int(s))
    return gt, known

def metrics(ranked, gt, ks):
    hit = {k: [] for k in ks}; rec = {k: [] for k in ks}; mrr = []
    for u, pos in gt.items():
        if u not in ranked:
            continue
        c = ranked[u]
        for k in ks:
            tk = set(c[:k]); hit[k].append(1.0 if tk & pos else 0.0); rec[k].append(len(tk & pos)/len(pos))
        rr = 0.0
        for i, e in enumerate(c, 1):
            if e in pos: rr = 1/i; break
        mrr.append(rr)
    out = {"MRR": np.mean(mrr) if mrr else 0.0}
    for k in ks:
        out[f"R@{k}"] = np.mean(rec[k]) if rec[k] else 0.0
    return out

In [ ]:
def candidate_generation(tag, ks):
    emb = load_emb(tag)
    gt, known = load_gt_known()
    user_vecs = emb[:n_users]
    index = faiss.IndexFlatIP(emb.shape[1]); index.add(user_vecs)
    max_k = max(ks)

    # unimodal
    uni = {}
    qs = [u for u in gt if u < n_users]
    Q = np.stack([user_vecs[u] for u in qs])
    _, I = index.search(Q, max_k + 50)
    for u, row in zip(qs, I):
        seen = known.get(u, set())
        uni[u] = [int(x) for x in row if x != u and x not in seen][:max_k]

    # кластеризуем businesses, юзер = смесь топ-m кластеров по engagement
    biz_vecs = emb[n_users:]
    nc = min(CFG["mixture_clusters"], n_biz)
    km = MiniBatchKMeans(n_clusters=nc, random_state=CFG["seed"], n_init=3)
    labels = km.fit_predict(biz_vecs)
    cent = km.cluster_centers_.astype("float32")
    cent /= np.linalg.norm(cent, axis=1, keepdims=True).clip(min=1e-12)
    biz_cluster = {n_users + i: int(c) for i, c in enumerate(labels)}

    eng = pd.concat([pd.read_csv(OUT / "review_train.tsv", sep="\t", header=None, names=["s", "r", "d"]),
                     pd.read_csv(OUT / "tip_train.tsv", sep="\t", header=None, names=["s", "r", "d"])])
    eng["c"] = eng["d"].map(biz_cluster); eng = eng.dropna()
    ucl = {}
    for u, c in zip(eng["s"].astype(int), eng["c"].astype(int)):
        ucl.setdefault(u, {}); ucl[u][c] = ucl[u].get(c, 0) + 1

    mix = {}
    for u in qs:
        seen = known.get(u, set()) | {u}
        counts = ucl.get(u)
        if not counts:
            mix[u] = uni[u]; continue
        top = sorted(counts.items(), key=lambda x: -x[1])[:CFG["mixture_top_m"]]
        tot = sum(w for _, w in top); merged = []
        for cl, w in top:
            b = max(1, round(max_k * w / tot))
            _, I = index.search(cent[cl][None, :], b + len(seen) + 1)
            for x in I[0]:
                x = int(x)
                if x not in seen and x not in merged:
                    merged.append(x)
                if len(merged) >= max_k: break
        mix[u] = merged[:max_k]

    # бейзлайны
    deg = pd.concat([pd.read_csv(OUT / "friend_train.tsv", sep="\t", header=None)[0],
                     pd.read_csv(OUT / "friend_train.tsv", sep="\t", header=None)[2]]).value_counts()
    popular = [int(x) for x in deg.index[:max_k * 3]]
    rng = np.random.default_rng(0)
    randl, popl = {}, {}
    for u in qs:
        seen = known.get(u, set()) | {u}
        randl[u] = [int(x) for x in rng.integers(0, n_users, max_k)]
        popl[u] = [x for x in popular if x not in seen][:max_k]

    return {"unimodal": metrics(uni, gt, ks), "mixture": metrics(mix, gt, ks),
            "popular": metrics(popl, gt, ks), "random": metrics(randl, gt, ks)}

res = candidate_generation(tag_ns, CFG["recall_ks"])
cg = pd.DataFrame(res).T[[f"R@{k}" for k in CFG["recall_ks"]] + ["MRR"]]
print("Candidate generation (friend) — Table 1 analog")
(cg * [100, 100, 100, 1]).round(3)

Candidate generation (friend) — Table 1 analog


,R@10,R@20,R@50,MRR
unimodal,0.072,0.128,0.256,0.003
mixture,0.119,0.199,0.513,0.002
popular,0.674,1.213,2.681,0.009
random,0.003,0.006,0.012,0.000


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

def rce(y, p):
    eps = 1e-7; p = np.clip(p, eps, 1 - eps)
    ce = log_loss(y, p, labels=[0, 1])
    b = min(max(float(np.mean(y)), eps), 1 - eps)
    ref = -(b * np.log(b) + (1 - b) * np.log(1 - b))
    return 100 * (ref - ce) / ref

def read_edges(name):
    p = OUT / name
    if p.stat().st_size == 0:
        return pd.DataFrame(columns=[0, 1, 2])
    return pd.read_csv(p, sep="\t", header=None)

def engagement_ranking(tag):
    emb = load_emb(tag)
    biz_ids_arr = np.arange(n_users, n_ent)
    def make(df, seed):
        pos = df[[0, 2]].rename(columns={0: "u", 2: "b"}); pos["y"] = 1
        rng = np.random.default_rng(seed)
        nb = rng.choice(biz_ids_arr, len(pos)); nu = pos["u"].to_numpy()
        neg = pd.DataFrame({"u": nu, "b": nb, "y": 0})
        d = pd.concat([pos, neg], ignore_index=True)
        X = np.concatenate([emb[d["u"].to_numpy()], emb[d["b"].to_numpy()]], 1)
        return X, d["y"].to_numpy()
    tr_df, te_df = read_edges("review_train.tsv"), read_edges("review_test.tsv")
    if len(tr_df) == 0 or len(te_df) == 0:
        print("engagement: пустой review train/test — пропуск")
        return {"ROC-AUC": float("nan"), "PR-AUC": float("nan"), "RCE": float("nan")}
    Xtr, ytr = make(tr_df, 1)
    Xte, yte = make(te_df, 2)
    clf = LogisticRegression(max_iter=300).fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]
    return {"ROC-AUC": roc_auc_score(yte, p), "PR-AUC": average_precision_score(yte, p), "RCE": rce(yte, p)}

eng_res = engagement_ranking(tag_ns)
print("Engagement ranking (review):", {k: round(v, 4) for k, v in eng_res.items()})

Engagement ranking (review): {'ROC-AUC': np.float64(0.7038), 'PR-AUC': np.float64(0.7268), 'RCE': np.float64(-9.63)}


## 7. Сравнение со статьёй

Ожидания: mixture >> unimodal

In [19]:
u10, m10 = res["unimodal"]["R@10"], res["mixture"]["R@10"]
lift = 100 * (m10 - u10) / u10 if u10 > 0 else float("nan")
print(f"mixture vs unimodal R@10 lift: {lift:+.1f}%  (статья: ~+300-500%)")

pd.DataFrame([
    {"task": "CandGen R@10 unimodal", "paper": "0.58%", "ours": f"{u10*100:.2f}%"},
    {"task": "CandGen R@10 mixture", "paper": "3.70%", "ours": f"{m10*100:.2f}%"},
    {"task": "Engagement RCE (review)", "paper": ">0", "ours": f"{eng_res['RCE']:.2f}"},
    {"task": "Engagement ROC-AUC", "paper": "~0.58-0.60 (Table 3)", "ours": f"{eng_res['ROC-AUC']:.3f}"},
])

mixture vs unimodal R@10 lift: +65.5%  (статья: ~+300-500%)


,task,paper,ours
0,CandGen R@10 unimodal,0.58%,0.07%
1,CandGen R@10 mixture,3.70%,0.12%
2,Engagement RCE (review),>0,-9.63
3,Engagement ROC-AUC,~0.58-0.60 (Table 3),0.704


## 8. Ablation по типам рёбер (гетерогенность)

Обучаем модель на разных подграфах и сравниваем friend-retrieval

In [20]:
abl_rows = []
for ab in ["all", "review_friend", "friend_only", "no_tip"]:
    tag, _ = train(ablation=ab, loss_name="ns")
    r = candidate_generation(tag, CFG["recall_ks"])
    abl_rows.append({"ablation": ab, "R@10_uni": r["unimodal"]["R@10"] * 100,
                     "R@10_mix": r["mixture"]["R@10"] * 100, "MRR_uni": r["unimodal"]["MRR"]})
pd.DataFrame(abl_rows).round(3)

all__ns__frequency ep1:   0%|          | 0/968 [00:00<?, ?it/s]

  ep1 train_loss=1.1372  val_Recall@10=0.0280


all__ns__frequency ep2:   0%|          | 0/968 [00:00<?, ?it/s]

  ep2 train_loss=1.0880  val_Recall@10=0.0336


all__ns__frequency ep3:   0%|          | 0/968 [00:00<?, ?it/s]

...

no_tip__ns__frequency ep9:   0%|          | 0/922 [00:00<?, ?it/s]

  ep9 train_loss=1.0585  val_Recall@10=0.0290


no_tip__ns__frequency ep10:   0%|          | 0/922 [00:00<?, ?it/s]

  ep10 train_loss=1.0569  val_Recall@10=0.0318
  early stop на эпохе 10 (нет роста Recall 5 эпох)
  best val_Recall@10=0.0338


,ablation,R@10_uni,R@10_mix,MRR_uni
0,all,0.064,0.008,0.003
1,review_friend,0.059,0.049,0.002
2,friend_only,0.200,0.006,0.005
3,no_tip,0.067,0.086,0.002
